<a href="https://colab.research.google.com/github/alijon070/Zenmart/blob/master/data_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install bing-image-downloader

In [ ]:
%pip install icrawler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [icrawler]


In [ ]:
%pip install -U ddgs requests pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 34.4 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ddgs]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [ ]:
from bing_image_downloader import downloader

In [ ]:
!rm -rf /content/pilaf_images

In [ ]:
!rm -rf /content/MyDrive/food101_dataset/pilaf

In [ ]:
from ddgs import DDGS
from PIL import Image
from io import BytesIO
import requests
import os

folder = "/content/uzbek_plov"
os.makedirs(folder, exist_ok=True)

query = "Uzbek traditional plov rice dish"

results = DDGS().images(
    query,
    max_results=150
)

count = 0

for result in results:
    if count >= 100:
        break

    try:
        url = result["image"]

        response = requests.get(
            url,
            timeout=10,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()

        img = Image.open(BytesIO(response.content))
        img = img.convert("RGB")

        # Skip very small images
        if img.width < 200 or img.height < 200:
            continue

        count += 1

        img.save(
            f"{folder}/plov_{count:03d}.jpg",
            quality=90
        )

    except Exception as e:
        print("Skipped:", str(e)[:100])

print(f"Successfully downloaded {count} images")

Skipped: 400 Client Error: Bad Request for url: http://i0.wp.com/unlimited-recipes.com/wp-content/uploads/201
Skipped: 403 Client Error: Forbidden for url: https://www.tiktok.com/api/img/?itemId=7577101031729859862&loca
Successfully downloaded 32 images


In [ ]:
from ddgs import DDGS
from PIL import Image
from io import BytesIO
import requests
import os
import hashlib

folder = "/content/uzbek_plov"
os.makedirs(folder, exist_ok=True)

queries = [
    "Uzbek traditional plov food",
    "Uzbek osh rice dish",
    "Uzbekistan palov with carrots and meat",
    "Samarkand plov",
    "Tashkent wedding plov",
    "Uzbek pilaf in kazan",
    "O'zbek palov osh taomi",
    "плов узбекский с мясом и морковью"
]

# Load existing images and their hashes
existing_hashes = set()
count = 0

for filename in os.listdir(folder):
    path = os.path.join(folder, filename)

    if not os.path.isfile(path):
        continue

    try:
        with Image.open(path) as img:
            normalized = img.convert("RGB")
            existing_hashes.add(
                hashlib.sha256(normalized.tobytes()).hexdigest()
            )
            count += 1
    except Exception:
        pass

print(f"Existing images: {count}")

target = 100

for query in queries:
    if count >= target:
        break

    print(f"\nSearching: {query}")

    try:
        results = DDGS().images(
            query,
            max_results=100
        )
    except Exception as e:
        print("Search failed:", e)
        continue

    for result in results:
        if count >= target:
            break

        try:
            url = result.get("image")
            if not url:
                continue

            response = requests.get(
                url,
                timeout=10,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            response.raise_for_status()

            with Image.open(BytesIO(response.content)) as img:
                img = img.convert("RGB")

                if img.width < 200 or img.height < 200:
                    continue

                image_hash = hashlib.sha256(
                    img.tobytes()
                ).hexdigest()

                if image_hash in existing_hashes:
                    continue

                filename = f"plov_{count + 1:04d}.jpg"
                path = os.path.join(folder, filename)

                while os.path.exists(path):
                    count += 1
                    filename = f"plov_{count + 1:04d}.jpg"
                    path = os.path.join(folder, filename)

                img.save(path, quality=90)

                existing_hashes.add(image_hash)
                count += 1

                print(f"Downloaded: {count}/{target}")

        except Exception:
            continue

print(f"\nTotal images: {count}")

Existing images: 32

Searching: Uzbek traditional plov food
Downloaded: 33/100
Downloaded: 34/100
Downloaded: 35/100
Downloaded: 36/100
Downloaded: 37/100
Downloaded: 38/100
Downloaded: 39/100
Downloaded: 40/100
Downloaded: 41/100
Downloaded: 42/100
Downloaded: 43/100
Downloaded: 44/100
Downloaded: 45/100
Downloaded: 46/100
Downloaded: 47/100
Downloaded: 48/100
Downloaded: 49/100
Downloaded: 50/100
Downloaded: 51/100
Downloaded: 52/100
Downloaded: 53/100
Downloaded: 54/100
Downloaded: 55/100
Downloaded: 56/100
Downloaded: 57/100
Downloaded: 58/100
Downloaded: 59/100
Downloaded: 60/100
Downloaded: 61/100
Downloaded: 62/100
Downloaded: 63/100
Downloaded: 64/100

Searching: Uzbek osh rice dish
Downloaded: 65/100
Downloaded: 66/100
Downloaded: 67/100
Downloaded: 68/100
Downloaded: 69/100
Downloaded: 70/100
Downloaded: 71/100
Downloaded: 72/100
Downloaded: 73/100
Downloaded: 74/100
Downloaded: 75/100
Downloaded: 76/100
Downloaded: 77/100
Downloaded: 78/100
Downloaded: 79/100
Downloaded: 80/

In [ ]:
!pip install -q duckduckgo_search

In [ ]:
import os
import shutil
from google.colab import drive

# Remove local /content/drive directory if it exists to allow mounting
if os.path.exists('/content/drive'):
    try:
        shutil.rmtree('/content/drive')
        print("Cleared local '/content/drive' directory to allow mounting.")
    except Exception as e:
        print(f"Warning: Could not remove directory: {e}")

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os, shutil

source_dir = '/content/uzbek_shashlik'
target_dir = '/content/drive/MyDrive/food101_dataset/kebab'

os.makedirs(target_dir, exist_ok=True)

for filename in os.listdir(source_dir):
    src_file = os.path.join(source_dir, filename)
    dst_file = os.path.join(target_dir, filename)
    shutil.move(src_file, dst_file)

In [ ]:
from ddgs import DDGS
from PIL import Image
from io import BytesIO
import requests
import os
import hashlib

# Folder for Uzbek shashlik images
folder = "/content/uzbek_shashlik"
os.makedirs(folder, exist_ok=True)

queries = [
    "Uzbek traditional shashlik kebab food",
    "Uzbek shashlik grilled meat skewers",
    "Uzbekistan traditional kebab dish",
    "Uzbek lamb shashlik",
    "Uzbek beef shashlik skewers",
    "Tashkent traditional shashlik",
    "Uzbek shashlik on charcoal grill",
    "Uzbek shashlik with onions",
    "O'zbek shashlik kabob taomi",
    "O'zbek qo'y go'shti shashlik",
    "O'zbek six kabob",
    "шашлык узбекский из баранины",
    "узбекский шашлык на мангале",
    "шашлык из говядины по-узбекски"
]

# Load existing images and hashes
existing_hashes = set()
count = 0

for filename in os.listdir(folder):
    path = os.path.join(folder, filename)

    if not os.path.isfile(path):
        continue

    try:
        with Image.open(path) as img:
            normalized = img.convert("RGB")
            existing_hashes.add(
                hashlib.sha256(normalized.tobytes()).hexdigest()
            )
            count += 1
    except Exception:
        pass

print(f"Existing images: {count}")

target = 100

for query in queries:
    if count >= target:
        break

    print(f"\nSearching: {query}")

    try:
        results = DDGS().images(
            query,
            max_results=100
        )
    except Exception as e:
        print("Search failed:", e)
        continue

    for result in results:
        if count >= target:
            break

        try:
            url = result.get("image")

            if not url:
                continue

            response = requests.get(
                url,
                timeout=10,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            response.raise_for_status()

            with Image.open(BytesIO(response.content)) as img:
                img = img.convert("RGB")

                # Skip small images
                if img.width < 200 or img.height < 200:
                    continue

                image_hash = hashlib.sha256(
                    img.tobytes()
                ).hexdigest()

                # Skip duplicate images
                if image_hash in existing_hashes:
                    continue

                filename = f"shashlik_{count + 1:04d}.jpg"
                path = os.path.join(folder, filename)

                while os.path.exists(path):
                    count += 1
                    filename = f"shashlik_{count + 1:04d}.jpg"
                    path = os.path.join(folder, filename)

                img.save(path, quality=90)

                existing_hashes.add(image_hash)
                count += 1

                print(f"Downloaded: {count}/{target}")

        except Exception:
            continue

print(f"\nTotal shashlik images: {count}")


Existing images: 0

Searching: Uzbek traditional shashlik kebab food
Downloaded: 1/100
Downloaded: 2/100
Downloaded: 3/100
Downloaded: 4/100
Downloaded: 5/100
Downloaded: 6/100
Downloaded: 7/100
Downloaded: 8/100
Downloaded: 9/100
Downloaded: 10/100
Downloaded: 11/100
Downloaded: 12/100
Downloaded: 13/100
Downloaded: 14/100
Downloaded: 15/100
Downloaded: 16/100
Downloaded: 17/100
Downloaded: 18/100
Downloaded: 19/100
Downloaded: 20/100
Downloaded: 21/100
Downloaded: 22/100
Downloaded: 23/100
Downloaded: 24/100
Downloaded: 25/100
Downloaded: 26/100
Downloaded: 27/100
Downloaded: 28/100
Downloaded: 29/100
Downloaded: 30/100
Downloaded: 31/100
Downloaded: 32/100
Downloaded: 33/100
Downloaded: 34/100

Searching: Uzbek shashlik grilled meat skewers
Downloaded: 35/100
Downloaded: 36/100
Downloaded: 37/100
Downloaded: 38/100
Downloaded: 39/100
Downloaded: 40/100
Downloaded: 41/100
Downloaded: 42/100
Downloaded: 43/100
Downloaded: 44/100
Downloaded: 45/100
Downloaded: 46/100
Downloaded: 47/100